## Requirments

In [1]:
%pip install langchain-google-genai dotenv

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.1.0-py3-none-any.whl.metadata (24 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.1.0-py3-none-any.whl (20 kB)

[notice] A new release of pip is available: 25.0 -> 25.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Load the API Key 

In [2]:
import getpass
import os
 
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# If the API key is not in environment variables, prompt the user
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API key: ")

## Simple Invoke Call or simple LLM Call with static Prompt

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
 
# Initialize model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)
 
# Simple invocation
messages = [
    ("system", "You are a helpful assistant that translates English to hinglish"),
    ("human", "Ameya made this small cheatsheet so that he can refer langchain code"),
]
response = llm.invoke(messages)
print(response.content)

Ameya ne yeh choti si cheatsheet banayi hai taaki woh langchain code ko refer kar sake.


what if you want to keep the translation from and to languages as variable we can do it with chains and prompt templates next example shows how can we do it

## Prompt Template and add Variables into Prompt

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
 
# Initialize model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
)
 
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that translates {input_language} to {output_language}."),
    ("human", "{input}"),
])
 
chain = prompt | llm
result = chain.invoke({
    "input_language": "English",
    "output_language": "hindi",
    "input": "Ameya Likes to play video games",
})
print(result.content)

अमेय को वीडियो गेम खेलना पसंद है।


## How Can i provide a image as an image to the AI bot and ask question about it ?

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
import base64
 
# Initialize model
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
 
# Using an image URL
message_url = HumanMessage(
    content=[
        {"type": "text", "text": "Which car is present in this image?"},
        {"type": "image_url", "image_url": "https://img.freepik.com/free-photo/mini-coupe-high-speed-drive-road-with-front-lights_114579-5040.jpg?t=st=1745855567~exp=1745859167~hmac=4e4390d04c1798c7c36ae40d23bebaa2559bbcde067b1e288c4335a72ae97dc5&w=996"},
    ]
)
result_url = llm.invoke([message_url])
print(result_url.content)

The car in the image is a Porsche.


In [9]:
# Using a local image
local_image_path = "images/powdery_mildew-min_480x480.webp"
with open(local_image_path, "rb") as image_file:
    encoded_image = base64.b64encode(image_file.read()).decode('utf-8')
 
message_local = HumanMessage(
    content=[
        {"type": "text", "text": "What is the issue with the leaf?"},
        {"type": "image_url", "image_url": f"data:image/webp;base64,{encoded_image}"}
    ]
)
result_local = llm.invoke([message_local])
print(result_local.content)

The leaf appears to have powdery mildew. The white, powdery spots are a characteristic symptom of this fungal disease.


## okay we can send images what about Audio ?

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
import base64
 
# Initialize model
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-exp")
 
audio_file_path = "audio/harvard.mp3"
audio_mime_type = "audio/mpeg"
 
with open(audio_file_path, "rb") as audio_file:
    encoded_audio = base64.b64encode(audio_file.read()).decode('utf-8')
 
message = HumanMessage(
    content=[
        {"type": "text", "text": "Transcribe this audio."},
        {"type": "media", "data": encoded_audio, "mime_type": audio_mime_type}
    ]
)
response = llm.invoke([message])
print(response.content)

The stale smell of old beer lingers.
It takes heat to bring out the odor.
A cold dip restores health and zest.
A salt pickle tastes fine with ham.
Tacos al pastor are my favorite.
A zestful food is the hot cross bun.


## How can we do the same for Video ?

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
import base64
 
# Initialize model
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
 
video_file_path = "../path/to/your/video/file.mp4"
video_mime_type = "video/mp4"
 
with open(video_file_path, "rb") as video_file:
    encoded_video = base64.b64encode(video_file.read()).decode('utf-8')
 
message = HumanMessage(
    content=[
        {"type": "text", "text": "Describe what's happening in this video."},
        {"type": "media", "data": encoded_video, "mime_type": video_mime_type}
    ]
)
response = llm.invoke([message])
print(response.content)

## Tool Calling or Function Calling

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
 
# Define a tool
@tool(description="Get the current weather in a given location")
def get_weather(location: str) -> str:
    return "It's sunny."
 
# Initialize model and bind the tool
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
llm_with_tools = llm.bind_tools([get_weather])
 
# Invoke with a query that should trigger the tool
query = "What's the weather in Pune?"
ai_msg = llm_with_tools.invoke(query)
 
# Access tool calls in the response
print(ai_msg.tool_calls)
 
# Pass tool results back to the model
tool_message = ToolMessage(
    content=get_weather(*ai_msg.tool_calls[0]['args']), 
    tool_call_id=ai_msg.tool_calls[0]['id']
)
final_response = llm_with_tools.invoke([ai_msg, tool_message])
print(final_response.content)

[{'name': 'get_weather', 'args': {'location': 'Pune'}, 'id': '8d169eb5-0503-4a8a-b1d8-1ab77d756b60', 'type': 'tool_call'}]
OK. It's sunny in Pune.


## Built-in Tools (Google Search, Code Execution)

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.ai.generativelanguage_v1beta.types import Tool as GenAITool
 
# Initialize model
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-thinking-exp-1219")
 
# Google Search
search_resp = llm.invoke(
    "When is the next total solar eclipse in Pune?",
    tools=[GenAITool(google_search={})],
)
print(search_resp.content)
 
# Code Execution
code_resp = llm.invoke(
    "What is 2*2, use python",
    tools=[GenAITool(code_execution={})],
)
 
for c in code_resp.content:
    if isinstance(c, dict):
        if c["type"] == 'code_execution_result':
            print(f"Code execution result: {c['code_execution_result']}")
        elif c["type"] == 'executable_code':
            print(f"Executable code: {c['executable_code']}")
    else:
        print(c)

While Pune, India will experience some solar and lunar eclipses in the coming years, a total solar eclipse is not expected to be visible from Pune in the near future.

Here's a summary of upcoming eclipses visible in Pune:
*   **Partial Solar Eclipse:** August 2, 2027.
*   **Lunar Eclipses:** Several lunar eclipses are expected, including a Total Lunar Eclipse on September 7, 2025, and another on December 31, 2028.

Total solar eclipses occur when the moon completely blocks the sun, and they are only visible from a narrow path on Earth. Future total solar eclipses are predicted for other parts of the world in the coming years, including locations in Africa, Europe, the Middle East, Australia, and North America.

While there will be solar eclipses in 2025, including a partial and an annular solar eclipse, neither will be visible from India.
Executable code: print(2 * 2)

Code execution result: 4

2 times 2 is 4.thought
The user is asking for the result of the multiplication 2*2 and expl

## Google Gemini Embeddings with LangChain

In [14]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
 
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-exp-03-07")
 
# Embed a single query
vector = embeddings.embed_query("hello, world!")
 
# Embed multiple documents
vectors = embeddings.embed_documents([
    "Today is Monday",
    "Today is Tuesday",
    "Today is April Fools day",
])

print(vector)
print(vectors)

[-0.024917153641581535, 0.012005362659692764, -0.003886754624545574, -0.05774897709488869, 0.0020742062479257584, 0.007751199882477522, -0.012343374080955982, -0.0027870230842381716, 0.02169838361442089, 0.0018268938874825835, -0.006741716060787439, -0.012518432922661304, -0.00817152950912714, 0.011904534883797169, 0.11910925805568695, 0.0015453619416803122, 0.02985035441815853, -0.030769294127821922, 0.005451818462461233, -6.30090944468975e-05, 0.0014195527182891965, 0.0024436695966869593, 0.021900925785303116, -0.006178081966936588, 0.00976465456187725, 0.018543638288974762, 0.04268296808004379, -0.017208943143486977, 0.021735263988375664, 0.00936521403491497, 0.007151819299906492, 0.021175969392061234, -0.03726310282945633, 0.005857192911207676, 0.006306711584329605, 0.028785381466150284, 0.004962958861142397, -0.0037474327255040407, -0.0033726205583661795, 0.0032673373352736235, -0.011026186868548393, -0.009033792652189732, 0.005852456670254469, -0.00579723808914423, 0.002631655428

## Using a Vector Store

In [17]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
 
# Initialize embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-exp-03-07")
 
text = "LangChain is the framework for building context-aware reasoning applications"
 
# Create vector store and retriever
vectorstore = InMemoryVectorStore.from_texts([text], embedding=embeddings)
retriever = vectorstore.as_retriever()
 
# Retrieve similar documents
retrieved_documents = retriever.invoke("What is LangChain?")
print(retrieved_documents[0].page_content)

LangChain is the framework for building context-aware reasoning applications


## Task Types for Retrival

In [19]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
 
# Different task types for different use cases
query_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-exp-03-07", 
    task_type="RETRIEVAL_QUERY"  # For queries
)
doc_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-exp-03-07", 
    task_type="RETRIEVAL_DOCUMENT"  # For documents
)
 
# Compare similarity
q_embed = query_embeddings.embed_query("What is the capital of France?")
d_embed = doc_embeddings.embed_documents(["The capital of France is Paris.", "Philipp likes to eat pizza."])
 
for i, d in enumerate(d_embed):
    similarity = cosine_similarity([q_embed], [d])[0][0]
    print(f"Document {i+1} similarity: {similarity}")

Document 1 similarity: 0.7892893360164779
Document 2 similarity: 0.5410037458373438
